# Langham Hotel Management System
Pranish Mahato | 850008479
Adapted from the supplied Python reference. Run the code cell and enter menu responses. Saved files use the notebook working directory.

In [ ]:
"""
LANGHAM HOTEL MANAGEMENT SYSTEM (LHMS) - Python version
Course : BBIM502 Introduction to Programming - Assessment 2
Student: Pranish Mahato | ID: 850008479

Console application: rooms, allocation, billing and file I/O with
exception handling. Refined from the supplied Python reference implementation.
"""

import os
import sys
from datetime import datetime

import math
from pathlib import Path

STUDENT_ID = "850008479"
MAIN_FILE = f"LHMS_{STUDENT_ID}.txt"
LINE = "*" * 70

ROOM_TYPES = {                       # type -> (nightly rate NZD, features)
    "1": ("Standard", 180.00, "1 Queen bed, city view, Wi-Fi"),
    "2": ("Deluxe",   260.00, "1 King bed, harbour view, minibar, Wi-Fi"),
    "3": ("Suite",    450.00, "2 rooms, King bed, lounge, bathtub, butler"),
}
GST = 0.15

rooms = []        # list of dicts: number, type, rate, features, allocated
allocations = []  # list of dicts: room_no, cust_no, cust_name, nights


# ----------------------------------------------------------------- helpers
def read_input(prompt):
    """input() wrapper that handles EOFError (e.g. Ctrl+Z / piped input)."""
    try:
        return input(prompt).strip()
    except EOFError:
        print("\nEnd of input reached. Exiting safely.")
        sys.exit(0)


def read_int(prompt, low=None, high=None):
    """Keep asking until a valid integer in range is entered (ValueError)."""
    while True:
        text = read_input(prompt)
        try:
            value = int(text)
            if (low is not None and value < low) or (high is not None and value > high):
                print(f"Enter a whole number in range: minimum {low}, maximum {high if high is not None else 'unlimited'}.")
                continue
            return value
        except ValueError:
            print(f"'{text}' is not a valid whole number. Please try again.")


def read_float(prompt, low=0.0, high=None):
    while True:
        try:
            value = float(read_input(prompt))
            if not math.isfinite(value) or value < low or (high is not None and value > high):
                print(f"Enter a finite number from {low} to {high if high is not None else 'unlimited'}.")
                continue
            return value
        except ValueError:
            print("Invalid number. Please try again.")


def find_room(number):
    for room in rooms:
        if room["number"] == number:
            return room
    return None


def find_allocation(number):
    for alloc in allocations:
        if alloc["room_no"] == number:
            return alloc
    return None


def header(title):
    print(f"\n{LINE}\n  {title}\n{LINE}")


# ------------------------------------------------------------ menu options
def add_rooms():
    header("ADD ROOMS")
    count = read_int("How many rooms would you like to add?: ", 1, 500)
    for i in range(1, count + 1):
        while True:
            number = read_int(f"Enter room number {i}: ", 1, 9999)
            if find_room(number):
                print("Same room number already exists. Try another.")
            else:
                break
        print("Room types: 1) Standard  2) Deluxe  3) Suite")
        choice = read_int("Select room type (1-3): ", 1, 3)
        rtype, rate, features = ROOM_TYPES[str(choice)]
        rooms.append({"number": number, "type": rtype, "rate": rate,
                      "features": features, "allocated": False})
        print(f"Room {number} ({rtype}) added.")
    print(f"Hotel now has {len(rooms)} room(s) in total.")


def delete_rooms():
    header("DELETE ROOMS")
    if not rooms:
        print("No rooms to delete. Please add rooms first.")
        return
    count = read_int("How many rooms would you like to delete?: ", 1, len(rooms))
    for i in range(1, count + 1):
        number = read_int(f"Enter room number {i} to delete: ", 1, 9999)
        room = find_room(number)
        if room is None:
            print("Could not find matching room number.")
        elif room["allocated"]:
            print("Room is currently occupied. De-allocate it first.")
        else:
            if read_input("Type YES to delete this room: ").upper() == "YES":
                rooms.remove(room)
                print(f"Room {number} deleted.")
            else:
                print("Deletion cancelled.")


def display_rooms():
    header("DISPLAY ROOM DETAILS")
    if not rooms:
        print("No rooms to display. Please add rooms first.")
        return
    print(f"{'Room':<8}{'Type':<10}{'Rate/Night':<14}{'Status':<12}Features")
    for r in sorted(rooms, key=lambda x: x["number"]):
        status = "Occupied" if r["allocated"] else "Available"
        print(f"{r['number']:<8}{r['type']:<10}${r['rate']:<13.2f}{status:<12}{r['features']}")


def allocate_rooms():
    header("ALLOCATE ROOMS")
    available = [r for r in rooms if not r["allocated"]]
    if not available:
        print("No available rooms. Add rooms or de-allocate first.")
        return
    count = read_int(f"How many rooms would you like to allocate? (1-{len(available)}): ",
                     1, len(available))
    done = 0
    while done < count:
        number = read_int(f"Room allocation {done + 1}: enter room number: ", 1, 9999)
        room = find_room(number)
        if room is None:
            print("Could not find matching room number. Enter a correct room number.")
            continue
        if room["allocated"]:
            print(f"Room {number} is already occupied. Choose another.")
            continue
        cust_no = read_int("Enter customer number: ", 1)
        cust_name = read_input("Enter customer name: ")
        if not cust_name:
            print("Customer name cannot be empty.")
            continue
        nights = read_int("Number of nights: ", 1, 365)
        room["allocated"] = True
        allocations.append({"room_no": number, "cust_no": cust_no,
                            "cust_name": cust_name, "nights": nights})
        print("Allocation has been done.")
        done += 1


def display_allocations():
    header("ROOM ALLOCATION DETAILS")
    if not allocations:
        print("No allocated rooms to display. Please allocate rooms first.")
        return
    for a in allocations:
        print(f"Room Number: {a['room_no']}  Customer Number: {a['cust_no']}  "
              f"Customer Name: {a['cust_name']}  Nights: {a['nights']}")


def calculate_bill(rate, nights, discount_pct):
    """Return (subtotal, discount, gst, total). Raises on bad values."""
    if isinstance(nights, bool) or not isinstance(nights, int):
        raise TypeError("Nights must be an integer.")
    if nights < 1 or nights > 365:
        raise ValueError("Nights must be between 1 and 365.")
    for value in (rate, discount_pct):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("Rate and discount must be numeric.")
        if not math.isfinite(value):
            raise ValueError("Rate and discount must be finite.")
    if rate <= 0 or not 0 <= discount_pct <= 100:
        raise ValueError("Rate must be positive; discount must be 0 to 100.")
    subtotal = rate * nights
    discount = subtotal * discount_pct / 100
    gst = (subtotal - discount) * GST
    total = subtotal - discount + gst
    if not math.isfinite(total):
        raise OverflowError("Bill total exceeds the numeric limit.")
    return subtotal, discount, gst, total


def billing_and_deallocation():
    header("BILLING & DE-ALLOCATION")
    if not allocations:
        print("No allocated rooms. Please allocate rooms first.")
        return
    number = read_int("Enter room number to bill and de-allocate: ", 1, 9999)
    alloc = find_allocation(number)
    room = find_room(number)
    if alloc is None or room is None:
        print("Could not find an allocation for that room.")
        return
    discount_pct = read_float("Discount percentage (0-100): ", 0.0, 100.0)
    try:
        if discount_pct > 100:
            raise ValueError("Discount cannot exceed 100%.")
        subtotal, discount, gst, total = calculate_bill(room["rate"], alloc["nights"], discount_pct)
        per_night = total / alloc["nights"]          # ZeroDivisionError if nights == 0
    except ZeroDivisionError:
        print("Error: number of nights is zero; cannot compute bill.")
        return
    except OverflowError as err:
        print(f"Calculation overflow: {err}")
        return
    except ValueError as err:
        print(f"Invalid billing value: {err}")
        return
    except TypeError:
        print("Billing data has an incorrect type.")
        return

    print(f"\n----------- INVOICE: LANGHAM HOTELS -----------")
    print(f"Customer : {alloc['cust_name']} (No. {alloc['cust_no']})")
    print(f"Room     : {number} ({room['type']})  x {alloc['nights']} night(s) @ ${room['rate']:.2f}")
    print(f"Subtotal : ${subtotal:,.2f}")
    print(f"Discount : -${discount:,.2f} ({discount_pct:g}%)")
    print(f"GST 15%  : ${gst:,.2f}")
    print(f"TOTAL    : ${total:,.2f}   (avg ${per_night:,.2f}/night)")
    print("-----------------------------------------------")

    if read_input("Type YES to confirm payment and checkout: ").upper() != "YES":
        print("Checkout cancelled. Allocation retained.")
        return
    allocations.remove(alloc)
    room["allocated"] = False
    print(f"Room {number} has been de-allocated and is available again.")


# ------------------------------------------------------------- file I/O
def save_allocations():
    header("SAVE ROOM ALLOCATIONS TO A FILE")
    if not allocations:
        print("Nothing to save. Please allocate rooms first.")
        return
    try:
        with open(MAIN_FILE, "a", encoding="utf-8") as f:
            f.write(LINE + "\n")
            f.write(f"Saved on: {datetime.now():%d/%m/%Y %H:%M:%S}\n")
            for a in allocations:
                f.write(f"Room Number: {a['room_no']} | Customer Number: {a['cust_no']} | "
                        f"Customer Name: {a['cust_name']} | Nights: {a['nights']}\n")
        print(f"Saved to '{os.path.abspath(MAIN_FILE)}'.")
    except (IOError, OSError) as err:                # IOError
        print(f"File unable to write: {err}")


def show_allocations_from_file():
    header("SHOW ROOM ALLOCATIONS FROM A FILE")
    try:
        with open(MAIN_FILE, "r", encoding="utf-8") as f:
            content = f.read()
        print(content if content.strip() else "File is empty.")
    except FileNotFoundError:                        # FileNotFoundError
        print(f"'{MAIN_FILE}' not found. Use option 7 (save) first.")
    except (OSError, UnicodeError) as err:
        print(f"Could not read file: {err}")


def backup_and_clear():
    header("BACKUP AND CLEAR ROOM ALLOCATION FILE")
    source = Path(MAIN_FILE)
    stamp = datetime.now().strftime("%d%m%Y_%H%M%S_%f")
    backup = source.with_name(f"LHMS_{STUDENT_ID}_Backup_{stamp}.txt")
    try:
        content = source.read_bytes()
        if not content:
            print("File is empty. Nothing to back up.")
            return
        if read_input("Type YES to back up and clear the saved file: ").upper() != "YES":
            print("Backup cancelled.")
            return
        with backup.open("xb") as dst:
            dst.write(content)
        if backup.read_bytes() != content:
            raise OSError("Backup verification failed; source retained.")
        if source.read_bytes() != content:
            raise OSError("Source changed during backup; source retained.")
        source.write_bytes(b"")
        print(f"Backed up to '{backup.resolve()}' and cleared '{source}'.")
        print("Live room allocations remain unchanged.")
    except FileNotFoundError:
        print(f"'{MAIN_FILE}' not found. Nothing to back up.")
    except OSError as err:
        print(f"Backup failed: {err}")


# ----------------------------------------------- exception demonstrations
def exception_demo():
    """Menu option 10: safely triggers each required built-in exception."""
    header("EXCEPTION HANDLING DEMONSTRATION")

    try:
        compile("rooms ==== 1", "<demo>", "exec")
    except SyntaxError as e:
        print("SyntaxError caught:", e.msg)
    try:
        int("abc")
    except ValueError as e:
        print("ValueError caught:", e)
    try:
        1 / 0
    except ZeroDivisionError as e:
        print("ZeroDivisionError caught:", e)
    try:
        [][5]
    except IndexError as e:
        print("IndexError caught:", e)
    try:
        print(undefined_variable)                    # noqa: F821
    except NameError as e:
        print("NameError caught:", e)
    try:
        "Room " + 101
    except TypeError as e:
        print("TypeError caught:", e)
    try:
        2.0 ** 100000
    except OverflowError as e:
        print("OverflowError caught:", e)
    try:
        open(os.getcwd(), "r")                       # a directory cannot be opened as a file
    except (IOError, OSError) as e:
        print("IOError caught:", type(e).__name__)
    try:
        import not_a_real_module                     # noqa: F401
    except ImportError as e:
        print("ImportError caught:", e)
    try:
        raise EOFError("input() hit end of file")
    except EOFError as e:
        print("EOFError caught:", e)
    try:
        raise FileNotFoundError(2, "Demonstration missing file", "missing_file_xyz.txt")
    except FileNotFoundError as e:
        print("FileNotFoundError caught:", e.filename)


# ------------------------------------------------------------------ menu
def show_menu():
    print(f"""
{LINE}
                LANGHAM HOTEL MANAGEMENT SYSTEM
                             MENU
0. Exit
1. Add Rooms
2. Delete Rooms
3. Display Room Details
4. Allocate Rooms
5. Display Room Allocation Details
6. Billing & De-Allocation
7. Save the Room Allocations To a File
8. Show the Room Allocations From a File
9. Backup and Clear Allocation File
10. Exception Handling Demonstration
{LINE}""")


ACTIONS = {
    1: add_rooms, 2: delete_rooms, 3: display_rooms, 4: allocate_rooms,
    5: display_allocations, 6: billing_and_deallocation,
    7: save_allocations, 8: show_allocations_from_file, 9: backup_and_clear,
    10: exception_demo,
}


def main():
    while True:
        show_menu()
        try:
            choice = read_int("Enter Your Choice Number Here (0-10): ", 0, 10)
            if choice == 0:
                print("Live rooms and allocations will be lost on exit. Option 7 saves text only.")
                if read_input("Type YES to exit: ").upper() == "YES":
                    print("Thank you for using Langham Hotel Management System. Goodbye!")
                    break
                continue
            ACTIONS[choice]()
        except KeyboardInterrupt:
            print("\nOperation cancelled.")
        except Exception as err:                     # last-resort safety net
            print(f"Unexpected error: {type(err).__name__}: {err}")


if __name__ == "__main__":
    main()
